# 04 Model Upgrade: Forecast Method Comparison

This notebook upgrades the baseline forecast analysis by comparing:

1. 7-day moving-average forecast  
2. 28-day moving-average forecast  
3. Ridge regression model  
4. HistGradientBoosting model  

The goal is not to chase Kaggle leaderboard performance. The goal is to evaluate whether more advanced, explainable models improve forecast accuracy and reduce forecast bias in a business-readable way.

**Important:** This notebook uses features available before or during the forecast period, such as item/store/category identifiers, calendar fields, price, event flags, SNAP flags, and pre-holdout historical rolling averages. This avoids using future actual demand as an input.

## 1. Setup

Run this notebook from the `notebooks/` folder.

Expected processed data file:

```text
../data/processed/foods_top100_recent365_clean.csv
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 100)

PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(PROCESSED_DIR / "foods_top100_recent365_clean.csv")
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(["id", "date"]).copy()

df.shape

## 2. Create Train / Test Split

We use the final 28 days as a holdout period.

This simulates a short-term planning window where we compare forecast methods against actual demand.

In [ ]:
max_date = df["date"].max()
test_start = max_date - pd.Timedelta(days=27)

train = df[df["date"] < test_start].copy()
test = df[df["date"] >= test_start].copy()

print("Train period:", train["date"].min(), "to", train["date"].max())
print("Test period:", test["date"].min(), "to", test["date"].max())
print("Train shape:", train.shape)
print("Test shape:", test.shape)

## 3. Build Historical Rolling Features

For training rows, rolling features are calculated using only prior observations.

For test rows, rolling features are frozen using the last available training-period history. This keeps the comparison fair because the model does not use future actual sales inside the holdout period.

In [ ]:
def add_training_rolling_features(data):
    data = data.sort_values(["id", "date"]).copy()

    data["rolling_7_mean"] = (
        data.groupby("id")["units_sold"]
        .transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
    )

    data["rolling_28_mean"] = (
        data.groupby("id")["units_sold"]
        .transform(lambda s: s.shift(1).rolling(28, min_periods=7).mean())
    )

    data["lag_7"] = data.groupby("id")["units_sold"].shift(7)
    data["lag_28"] = data.groupby("id")["units_sold"].shift(28)

    return data

train_model = add_training_rolling_features(train)

# Drop early rows where rolling history is not available
train_model = train_model.dropna(subset=["rolling_7_mean", "rolling_28_mean", "lag_7", "lag_28"]).copy()

train_model.shape

In [ ]:
# Static pre-holdout historical features for the test period
hist_features = (
    train.sort_values(["id", "date"])
    .groupby("id")
    .agg(
        rolling_7_mean=("units_sold", lambda s: s.tail(7).mean()),
        rolling_28_mean=("units_sold", lambda s: s.tail(28).mean()),
        lag_7=("units_sold", lambda s: s.iloc[-7] if len(s) >= 7 else np.nan),
        lag_28=("units_sold", lambda s: s.iloc[-28] if len(s) >= 28 else np.nan)
    )
    .reset_index()
)

test_model = test.drop(columns=["rolling_7_mean", "rolling_28_mean", "lag_7", "lag_28"], errors="ignore")
test_model = test_model.merge(hist_features, on="id", how="left")

test_model[["id", "date", "units_sold", "rolling_7_mean", "rolling_28_mean", "lag_7", "lag_28"]].head()

## 4. Baseline Forecasts

The baseline methods are simple and business-readable:

- 7-day historical average
- 28-day historical average

In [ ]:
forecast_compare = test_model.copy()

forecast_compare["forecast_7day_avg"] = forecast_compare["rolling_7_mean"]
forecast_compare["forecast_28day_avg"] = forecast_compare["rolling_28_mean"]

forecast_compare[["id", "date", "units_sold", "forecast_7day_avg", "forecast_28day_avg"]].head()

## 5. Prepare Features for Model Training

Features include:

- Product / department / store identifiers
- Calendar features
- Event and SNAP flags
- Price
- Historical rolling averages and lag features

The target variable is `units_sold`.

In [ ]:
target_col = "units_sold"

categorical_cols = [
    "item_id", "dept_id", "store_id", "state_id", "weekday"
]

numeric_cols = [
    "sell_price", "month", "wday",
    "has_event", "is_weekend", "snap_flag",
    "rolling_7_mean", "rolling_28_mean",
    "lag_7", "lag_28"
]

X_train = train_model[categorical_cols + numeric_cols].copy()
y_train = train_model[target_col].copy()

X_test = test_model[categorical_cols + numeric_cols].copy()
y_test = test_model[target_col].copy()

X_train.shape, X_test.shape

## 6. Model 1: Ridge Regression

Ridge regression is a simple, explainable benchmark. It can capture category, store, calendar, price, and historical average effects, but it remains relatively easy to explain.

In [ ]:
ridge_preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False))
        ]), numeric_cols)
    ]
)

ridge_model = Pipeline(steps=[
    ("preprocess", ridge_preprocess),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

forecast_compare["forecast_ridge"] = ridge_model.predict(X_test)
forecast_compare["forecast_ridge"] = forecast_compare["forecast_ridge"].clip(lower=0)

forecast_compare[["id", "date", "units_sold", "forecast_ridge"]].head()

## 7. Model 2: HistGradientBoosting

This tree-based model can capture non-linear relationships and interactions, such as how demand patterns differ by item, store, weekday, event, and price.

It is more flexible than linear regression but still available in scikit-learn without installing extra packages.

In [ ]:
hgb_preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), categorical_cols),
        ("num", SimpleImputer(strategy="median"), numeric_cols)
    ],
    remainder="drop"
)

hgb_model = Pipeline(steps=[
    ("preprocess", hgb_preprocess),
    ("model", HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=42
    ))
])

hgb_model.fit(X_train, y_train)

forecast_compare["forecast_hgb"] = hgb_model.predict(X_test)
forecast_compare["forecast_hgb"] = forecast_compare["forecast_hgb"].clip(lower=0)

forecast_compare[["id", "date", "units_sold", "forecast_hgb"]].head()

## 8. Compare Forecast Accuracy and Bias

Definitions:

- `error = forecast - actual`
- Negative bias means under-forecasting
- Positive bias means over-forecasting

In [ ]:
def metrics_table(data, forecast_cols, actual_col="units_sold"):
    rows = []

    actual = data[actual_col]

    for col in forecast_cols:
        forecast = data[col]
        error = forecast - actual

        mae = mean_absolute_error(actual, forecast)
        rmse = np.sqrt(mean_squared_error(actual, forecast))
        bias = error.sum() / actual.sum()
        smape = np.mean(
            2 * np.abs(forecast - actual) /
            (np.abs(actual) + np.abs(forecast) + 1e-9)
        )

        rows.append({
            "method": col,
            "MAE": mae,
            "RMSE": rmse,
            "Forecast Bias": bias,
            "sMAPE": smape
        })

    return pd.DataFrame(rows).sort_values("MAE")

forecast_cols = [
    "forecast_7day_avg",
    "forecast_28day_avg",
    "forecast_ridge",
    "forecast_hgb"
]

model_comparison = metrics_table(forecast_compare, forecast_cols)

model_comparison

In [ ]:
model_comparison.to_csv(REPORTS_DIR / "model_comparison_metrics.csv", index=False)
print("Saved:", REPORTS_DIR / "model_comparison_metrics.csv")

## 9. Actual vs Forecast Trend

This chart aggregates item-store forecasts to the daily level.

In [ ]:
daily_model_forecast = (
    forecast_compare.groupby("date", as_index=False)
    .agg(
        actual_units=("units_sold", "sum"),
        forecast_7day_units=("forecast_7day_avg", "sum"),
        forecast_28day_units=("forecast_28day_avg", "sum"),
        forecast_ridge_units=("forecast_ridge", "sum"),
        forecast_hgb_units=("forecast_hgb", "sum")
    )
)

daily_model_forecast.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily_model_forecast["date"], daily_model_forecast["actual_units"], label="Actual")
plt.plot(daily_model_forecast["date"], daily_model_forecast["forecast_7day_units"], label="7-Day Avg")
plt.plot(daily_model_forecast["date"], daily_model_forecast["forecast_28day_units"], label="28-Day Avg")
plt.plot(daily_model_forecast["date"], daily_model_forecast["forecast_ridge_units"], label="Ridge")
plt.plot(daily_model_forecast["date"], daily_model_forecast["forecast_hgb_units"], label="HistGradientBoosting")
plt.title("Actual vs Forecast by Method - 28-Day Holdout")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Model-Based Under-Forecast and Over-Forecast Risk

Use the best-performing method by MAE for exception analysis.

If the tree-based model performs best, use `forecast_hgb`. Otherwise, change `best_forecast_col` manually.

In [ ]:
best_forecast_col = model_comparison.iloc[0]["method"]
print("Best forecast method by MAE:", best_forecast_col)

forecast_compare["best_forecast"] = forecast_compare[best_forecast_col]
forecast_compare["best_error"] = forecast_compare["best_forecast"] - forecast_compare["units_sold"]
forecast_compare["best_abs_error"] = forecast_compare["best_error"].abs()

In [ ]:
model_item_summary = (
    forecast_compare.groupby(["item_id", "dept_id"], as_index=False)
    .agg(
        actual_units=("units_sold", "sum"),
        forecast_units=("best_forecast", "sum"),
        mae=("best_abs_error", "mean")
    )
)

model_item_summary["forecast_error"] = (
    model_item_summary["forecast_units"] - model_item_summary["actual_units"]
)

model_item_summary["forecast_bias_pct"] = (
    model_item_summary["forecast_error"] / model_item_summary["actual_units"]
)

model_under_forecast_items = model_item_summary.sort_values("forecast_error").head(20)
model_over_forecast_items = model_item_summary.sort_values("forecast_error", ascending=False).head(20)

display(model_under_forecast_items)
display(model_over_forecast_items)

In [ ]:
daily_model_forecast.to_csv(REPORTS_DIR / "daily_model_actual_vs_forecast.csv", index=False)
forecast_compare.to_csv(REPORTS_DIR / "forecast_method_comparison_detail.csv", index=False)
model_item_summary.to_csv(REPORTS_DIR / "model_item_forecast_summary.csv", index=False)
model_under_forecast_items.to_csv(REPORTS_DIR / "model_under_forecast_risk_items.csv", index=False)
model_over_forecast_items.to_csv(REPORTS_DIR / "model_over_forecast_risk_items.csv", index=False)

print("Model upgrade report tables saved.")

## 11. Business Interpretation Template

After running the notebook, update the placeholders below:

- The best-performing method by MAE was: `[method]`.
- Compared with the 28-day moving average baseline, the best model changed MAE from `[baseline MAE]` to `[model MAE]`.
- Forecast bias changed from `[baseline bias]` to `[model bias]`.
- If bias remains negative, the business should still monitor under-forecast risk and potential stockout exposure.
- If bias becomes positive, the business should monitor over-forecast risk and potential excess inventory.
- Even if the model improves accuracy, exception workflow is still needed because high-volume forecast errors can have large operational impact.

## Resume Bullet Upgrade

- Compared moving-average baselines with regression and tree-based forecasting models to evaluate improvements in forecast accuracy, bias, and item-level exception detection.
- Used historical rolling averages, price, calendar, event, SNAP, item, department, and store features to support explainable demand forecasting.